# 🌌 OMEGA v3: Orbital Maximum Engine for Galaxy Assault
**Competition: Orbit Wars — Kaggle Simulation | Public Score: 678.5**

OMEGA v3 is not just a script; it's a comprehensive 6-layer architecture capable of executing 12 different mission types simultaneously. It completely abandons the "naive capture" logic and replaces it with mathematical optimization, economic awareness, and precise orbital physics.

## 🧠 The 6-Layer Architecture
1. **Physics Layer:** Orbital prediction and sun avoidance (Iterative convergence).
2. **World Model:** Turn-by-turn timeline simulation with Binary Search.
3. **Economic Mode:** Dynamic scaling (EXPAND / BALANCED / AGGRO).
4. **Policy Builder:** Dynamic reserves and reaction times.
5. **Mission Engine:** 12 distinct mission types (Snipe, Swarm, Rescue, etc.) prioritized by ROI.
6. **Executor:** The Tsunami Strike and Doomed Evacuation engine.

## 1. Physics: The Tsunami Strike (Logarithmic Speed)
In Orbit Wars, fleet speed scales logarithmically:
`fleet_speed = 1.0 + 5.0 × (log(ships) / log(1000)) ^ 1.5`

A fleet of 100 ships is almost 4x faster than a fleet of 1 ship. OMEGA v3 doesn't just blindly send extra ships to move faster. It mathematically proves whether the **turns saved** are worth the **ships spent**, effectively calculating the ROI of velocity.

In [ ]:
def speed_optimal_send(needed, available, distance, prod_per_turn=2):
    """
    Mathematically justified Tsunami Strike.
    If sending more ships saves enough turns to justify the cost (via production gained), 
    we unleash a Tsunami wave. Otherwise, we send just enough.
    """
    if available <= 0 or needed <= 0: return needed
    if available < needed: return needed

    base_turns = max(1, math.ceil(distance / fleet_speed(max(1, needed))))

    # === FULL TSUNAMI ===
    if available >= needed * 1.8 and available >= 35:
        candidate  = min(available, max(needed, int(available * 0.87))) # Send 87% of budget
        cand_turns = max(1, math.ceil(distance / fleet_speed(max(1, candidate))))
        turns_saved = base_turns - cand_turns
        extra_ships = candidate - needed

        # Is the speed boost worth it?
        if turns_saved >= 2:
            return candidate # Mathematically justified!
            
        if extra_ships <= available * 0.45:
            return candidate # Cheap enough to justify

    # === SOFT TSUNAMI ===
    modest = min(available, int(needed * 1.20))
    if modest > needed:
        mod_turns = max(1, math.ceil(distance / fleet_speed(max(1, modest))))        
        if base_turns - mod_turns >= 1:
            return modest # +20% ships saves 1 turn

    return min(available, max(needed, int(needed * 1.05)))

## 2. Economic Modes (EXPAND / BALANCED / AGGRO)
Blindly expanding is correct when you are ahead, but fatal when you are behind. OMEGA evaluates the `eco_ratio = my_production / enemy_production`. 

If the ratio falls below 0.72 (`AGGRO` mode), it stops targeting neutral planets and mathematically multiplies the value of high-production enemy planets by up to **1.81x**, forcing aggressive economic denial.

In [ ]:
# --- EXCERPT: DYNAMIC ECONOMIC SCORING ---
def determine_eco_mode(my_prod, enemy_prod):
    ratio = my_prod / max(1, enemy_prod)
    if   ratio >= 1.35: return EcoMode.EXPAND,   ratio
    elif ratio <= 0.72: return EcoMode.AGGRO,    ratio
    else:               return EcoMode.BALANCED, ratio

def apply_eco_multipliers(base_val, target_owner, target_prod, player, eco_mode):
    val = base_val
    if eco_mode == EcoMode.EXPAND:
        if target_owner == -1: val *= 1.30 # Favor neutrals
        else:                  val *= 0.80 # Avoid costly fights
            
    elif eco_mode == EcoMode.AGGRO:
        if target_owner not in (-1, player):
            val *= 1.45 # Desperate attack on enemies
            if target_prod >= 4:
                val *= 1.25 # Destroy their best production!
        elif target_owner == -1:
            val *= 0.80 # Ignore neutrals
    return val

## 3. Timeline Simulation & Binary Search
When asked "How many ships do I need to survive an attack?", OMEGA doesn't guess. It runs a full timeline simulation. To find the exact minimum garrison needed without lagging the Kaggle environment, it uses a **Binary Search**, finding the exact threshold in `O(log N)` simulations.

In [ ]:
# --- EXCERPT: BINARY SEARCH TIMELINE RESOLUTION ---
# Binary search: minimum ships to keep to survive through horizon
keep_needed = 0

def survives(keep_ships):
    """Simulate the timeline if we keep 'keep_ships' as garrison."""
    so, sg = planet_owner, float(keep_ships)
    for t in range(1, horizon + 1):
        if so != -1: sg += planet_prod # Add production
        gr = arrivals_by_turn.get(t, [])
        if gr:
            so, sg = resolve_combat(so, sg, gr) # Resolve massive fleet battles
            if so != player: return False
    return so == player

# Binary Search O(log N)
if survives(int(planet_ships)):
    lo, hi = 0, int(planet_ships)
    while lo < hi:
        mid = (lo + hi) // 2
        if survives(mid): hi = mid
        else:             lo = mid + 1
    keep_needed = lo